<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/18_classification/18_5_MutliClassClassification/18_5_3_Imbalanced_Multiclass.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multiclass Classification: Part 3
## When One Class Is Rare

---

## What This Notebook Is About

The penguins were convenient. Three species, roughly balanced, and so easy to classify that a well-tuned model gets every single one right. That is a best-case scenario.

Real classification problems are rarely that clean.

This notebook works with a dataset where the rarest class is also the most dangerous. You will see how class imbalance distorts the metrics you learned in the last notebook — and why one metric in particular will actively mislead you.

**What you will learn:**
1. How to spot class imbalance and compute the "do nothing" baseline
2. Why weighted F1 can look acceptable while the model is quietly failing on the minority class
3. How to use macro F1 as the honest metric for imbalanced multiclass problems
4. How sample weights give rare classes equal influence during training
5. How to verify results honestly with stratified cross-validation

## The Problem: Fetal Health Monitoring

During pregnancy, doctors monitor the baby's heart rate and uterine contractions using cardiotocography (CTG). Based on 35 measurements — heart rate variability, acceleration patterns, deceleration counts, and more — each monitoring session is classified as:

- **Normal** — everything looks healthy
- **Suspect** — worth monitoring more closely
- **Pathological** — needs immediate medical attention

The dataset contains 2,126 CTG sessions from a medical study. Here is the complication:

Most pregnancies are uneventful. In this dataset:
- **Normal:** 78% of sessions
- **Suspect:** 14% of sessions
- **Pathological:** 8% of sessions

A model that reflexively says "Normal" for every session would be right 78% of the time — and would miss 100% of the dangerous cases.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (ConfusionMatrixDisplay, classification_report,
                              f1_score, recall_score, confusion_matrix)
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils import compute_sample_weight

sns.set_theme(style="whitegrid")

In [ ]:
# Load the Cardiotocography dataset (3-class NSP version) from OpenML
# version=2 contains Normal/Suspect/Pathological labels
data = fetch_openml(name="cardiotocography", version=2, as_frame=True, parser="auto")

df = data.frame.copy()

# Map the integer codes to descriptive labels
target_map = {"1": "Normal", "2": "Suspect", "3": "Pathological"}
df["Class"] = df["Class"].astype(str).map(target_map)

df = df.dropna().reset_index(drop=True)

class_names = ["Normal", "Suspect", "Pathological"]
label_to_int = {name: i for i, name in enumerate(class_names)}

X = df.drop(columns=["Class"]).to_numpy(dtype=float)
y = np.array([label_to_int[c] for c in df["Class"]])

print(f"Dataset: {X.shape[0]} sessions, {X.shape[1]} features")
print()
print("Class distribution:")
counts = np.bincount(y)
for name, count in zip(class_names, counts):
    print(f"  {name:<15}  {count:>5}  ({count/len(y):.1%})")

---

## Section 1: Measuring the Problem

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
bar_colors = ["#4C72B0", "#DD8452", "#C44E52"]
bars = ax.bar(class_names, counts, color=bar_colors, edgecolor="white")
for i, count in enumerate(counts):
    ax.text(i, count + 15, f"{count:,}\n({count/len(y):.1%})",
            ha="center", fontweight="bold")
ax.set_ylabel("Number of Sessions")
ax.set_title("CTG Class Distribution: Normal Dominates")
ax.set_ylim(0, max(counts) * 1.2)
plt.tight_layout()
plt.show()

naive_baseline = counts[0] / len(y)
print(f"Naive baseline (always predict 'Normal'): {naive_baseline:.1%}")
print()
print("A model that does absolutely nothing and always says 'Normal'")
print("would achieve this accuracy — and catch zero dangerous cases.")

That bar chart tells the whole story. Normal is eight times as common as Pathological. Any model that is not explicitly designed to treat rare classes as important will learn to ignore them.

Now let us see what actually happens.

---

## Section 2: The Imbalance Trap in Action

We will start with a deliberately simple model: a **Decision Tree with only two levels of depth**. A tree this shallow can only ask two yes/no questions about the data, so it has very limited capacity.

When capacity is limited, models gravitate toward the majority class. They get more reward (more correct predictions) from serving the large Normal class than from learning the rare Pathological cases.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# A very shallow tree: simple, limited, and prone to the imbalance trap
naive_model = DecisionTreeClassifier(max_depth=2, random_state=42)
naive_model.fit(X_train, y_train)
y_pred_naive = naive_model.predict(X_test)

print("Shallow Decision Tree (max_depth=2):")
print(classification_report(y_test, y_pred_naive, target_names=class_names, digits=3))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_naive,
    display_labels=class_names,
    cmap="Oranges",
    ax=ax
)
ax.set_title("Shallow Decision Tree (max_depth=2): Confusion Matrix", fontsize=12)
ax.grid(False)
plt.tight_layout()
plt.show()

### Reading the Trap

Look at the confusion matrix, row by row.

**Normal row:** Nearly everything correct. The model is good at Normal — because it has 1,655 examples to learn from and Normal is the "safe" default.

**Suspect row:** Look at how many actual Suspect sessions are mislabeled as Normal. The model does not know what to do with borderline cases, so it defaults to the majority.

**Pathological row:** How many Pathological sessions does the model catch? Each miss is a pregnancy needing intervention that gets waved through as "Normal."

Now look at the classification report:

- **Accuracy ≈ 0.94** — sounds excellent
- **Weighted F1 ≈ 0.90** — also looks acceptable
- **Macro F1 ≈ 0.80** — meaningfully lower

That gap is the warning sign. Weighted F1 is propped up by the 78% Normal class. Macro F1 gives Suspect and Pathological equal voice — and they are struggling.

---

## Section 3: Macro F1 vs. Weighted F1 — Why the Gap Matters

In [ ]:
# Demonstrate the gap numerically
f1_per_class = f1_score(y_test, y_pred_naive, average=None)
support       = np.bincount(y_test)

macro_f1    = f1_per_class.mean()
weighted_f1 = np.sum(f1_per_class * support) / support.sum()

print("Per-class F1 scores:")
for name, f, s in zip(class_names, f1_per_class, support):
    print(f"  {name:<15}  F1={f:.3f}  (support={s})")

print()
print(f"Macro F1   = ({' + '.join(f'{v:.3f}' for v in f1_per_class)}) / 3 = {macro_f1:.3f}")
print()
wterms = ' + '.join(f'{f1_per_class[i]:.3f}×{support[i]}' for i in range(3))
print(f"Weighted F1 = ({wterms}) / {support.sum()} = {weighted_f1:.3f}")
print()
print(f"Gap: weighted F1 - macro F1 = {weighted_f1 - macro_f1:.3f}")
print()
print("Interpretation: The gap reveals that the model performs very differently")
print("across classes. Weighted F1 hides the Pathological and Suspect failures")
print("behind the Normal class's strong performance.")

The gap here — roughly 0.10 — is the alarm. It says: "This model is doing well on Normal but substantially worse on the minority classes."

> **When classes are imbalanced and all classes matter, macro F1 is your headline metric. Weighted F1 will mislead you.**

This is the central lesson of this notebook. Every decision you make going forward — which model to choose, whether to use sample weights, whether a given performance is "good enough" — should be evaluated against macro F1, not weighted F1 and not accuracy.

---

## Section 4: XGBoost — Better by Design

XGBoost's ensemble approach (many trees, each correcting the last) naturally handles imbalanced data better than a single shallow tree. It can build specialized subtrees for rare classes. Let us see how much it helps without any explicit imbalance correction.

In [ ]:
model_plain = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    n_estimators=10,
    max_depth=2,
    learning_rate=0.5,
    random_state=42,
    eval_metric="mlogloss"
)
model_plain.fit(X_train, y_train)
y_pred_plain = model_plain.predict(X_test)

print("XGBoost (n_estimators=10, no weight adjustment):")
print(classification_report(y_test, y_pred_plain, target_names=class_names, digits=3))

Even a small ensemble of ten trees dramatically outperforms the single-level tree. Suspect recall jumps from below 60% to above 85%. Pathological recall was already high and stays high.

But notice: Suspect recall is still not perfect. The model is still somewhat biased toward Normal on borderline Suspect cases. That is where sample weights come in.

---

## Section 5: Sample Weights — Giving Rare Classes Equal Voice

The plain model's problem: it sees 1,655 Normal examples and only 295 Suspect examples in training. It gets five times as many opportunities to learn Normal as Suspect. Naturally, it becomes five times more confident about Normal.

**Sample weights** fix this by making each training example count in proportion to how rare its class is. A single Pathological example, weighted to match the frequency ratio, counts as much in the loss function as nine Normal examples.

The model now has equal incentive to classify each class correctly, regardless of how often it appears in the data.

In [ ]:
# Compute balanced weights: rare classes get high weight, common classes get low weight
sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)

# Show what the weights look like per class
print("Sample weights assigned:")
for i, name in enumerate(class_names):
    count = (y_train == i).sum()
    w = sample_weights[y_train == i][0]
    print(f"  {name:<15}  count={count:>4}  weight={w:.3f}")

print()
print("Intuition: Normal is 5.6× more common than Suspect,")
print("so each Suspect example gets 5.6× the weight of a Normal example.")

In [ ]:
model_balanced = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    n_estimators=10,
    max_depth=2,
    learning_rate=0.5,
    random_state=42,
    eval_metric="mlogloss"
)
model_balanced.fit(X_train, y_train, sample_weight=sample_weights)
y_pred_balanced = model_balanced.predict(X_test)

print("XGBoost + balanced sample weights:")
print(classification_report(y_test, y_pred_balanced, target_names=class_names, digits=3))

In [ ]:
# Side-by-side confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_plain,
    display_labels=class_names, cmap="Oranges", ax=axes[0]
)
axes[0].set_title("XGBoost (no weight adjustment)", fontsize=12)
axes[0].grid(False)

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_balanced,
    display_labels=class_names, cmap="Blues", ax=axes[1]
)
axes[1].set_title("XGBoost + balanced sample weights", fontsize=12)
axes[1].grid(False)

plt.suptitle("Effect of Sample Weights on the Confusion Matrix", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Direct metric comparison
macro_plain    = f1_score(y_test, y_pred_plain,    average="macro")
macro_balanced = f1_score(y_test, y_pred_balanced, average="macro")
weighted_plain    = f1_score(y_test, y_pred_plain,    average="weighted")
weighted_balanced = f1_score(y_test, y_pred_balanced, average="weighted")

rec_plain    = recall_score(y_test, y_pred_plain,    average=None)
rec_balanced = recall_score(y_test, y_pred_balanced, average=None)

print("Metric comparison: plain vs. balanced XGBoost\n")
header = "  {:<24}  {:>8}  {:>10}  {:>8}".format("Metric", "Plain", "Balanced", "Change")
print(header)
print("  " + "-" * (len(header) - 2))
print("  {:<24}  {:>8.3f}  {:>10.3f}  {:>+8.3f}".format(
    "Macro F1", macro_plain, macro_balanced, macro_balanced - macro_plain))
print("  {:<24}  {:>8.3f}  {:>10.3f}  {:>+8.3f}".format(
    "Weighted F1", weighted_plain, weighted_balanced, weighted_balanced - weighted_plain))
print()
for i, name in enumerate(class_names):
    label = f"Recall ({name})"
    print("  {:<24}  {:>8.3f}  {:>10.3f}  {:>+8.3f}".format(
        label, rec_plain[i], rec_balanced[i], rec_balanced[i] - rec_plain[i]))

The improvement from balanced weights is real but modest with this dataset — because XGBoost already handles the imbalance reasonably well with its ensemble structure.

Two observations worth internalizing:

1. **Macro F1 improved.** That is the metric we care about, and it went in the right direction.
2. **Weighted F1 also improved.** For once, both metrics agree on the direction, because the balanced model is genuinely better across the board.

In some datasets — particularly when the minority class is very rare (< 5%) and the features are noisy — sample weights make a dramatic difference. In others, like this one, the improvement is incremental. The principle remains valid even when the magnitude varies.

---

## Section 6: Cross-Validation for Honest Evaluation

A single train-test split might put most of the Pathological cases into training or testing by chance. With only 176 Pathological examples in the full dataset, this matters.

**Stratified K-Fold** solves it: each fold preserves the class proportions, so every fold has roughly the same fraction of Pathological cases. The average macro F1 across folds is a far more reliable estimate than a single split.

In [ ]:
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

plain_scores    = []
balanced_scores = []

for train_idx, test_idx in outer_cv.split(X, y):
    X_tr, X_te = X[train_idx], X[test_idx]
    y_tr, y_te = y[train_idx], y[test_idx]

    # Plain model
    m_plain = xgb.XGBClassifier(
        objective="multi:softprob", num_class=3, n_estimators=10,
        max_depth=2, learning_rate=0.5, random_state=42, eval_metric="mlogloss"
    )
    m_plain.fit(X_tr, y_tr)
    plain_scores.append(f1_score(y_te, m_plain.predict(X_te), average="macro"))

    # Balanced model
    w_tr = compute_sample_weight(class_weight="balanced", y=y_tr)
    m_bal = xgb.XGBClassifier(
        objective="multi:softprob", num_class=3, n_estimators=10,
        max_depth=2, learning_rate=0.5, random_state=42, eval_metric="mlogloss"
    )
    m_bal.fit(X_tr, y_tr, sample_weight=w_tr)
    balanced_scores.append(f1_score(y_te, m_bal.predict(X_te), average="macro"))

plain_scores    = np.array(plain_scores)
balanced_scores = np.array(balanced_scores)

print("5-fold CV macro F1 (stratified — class proportions preserved in each fold):\n")
print(f"  Plain XGBoost   : {plain_scores.mean():.3f} ± {plain_scores.std():.3f}")
print(f"  Balanced XGBoost: {balanced_scores.mean():.3f} ± {balanced_scores.std():.3f}")
print()
print("The ± value (standard deviation across folds) tells you how stable the estimate is.")
print("Low ± means the estimate is trustworthy. High ± means performance varies with the split.")

Cross-validation confirms what the single split showed. The balanced model is consistently better, and the result is not a lucky split.

Note that we used the exact same model configuration in both loops — only the sample weights differ. That is a clean controlled comparison.

---

## Section 7: Which Metric to Use?

| What you want to know | Use this metric |
|---|---|
| Overall honest performance with imbalanced classes | **Macro F1** |
| "Is the model catching the dangerous minority class?" | **Pathological recall** (per-class recall) |
| Performance weighted by how often each class appears | Weighted F1 |
| Is the model doing better than doing nothing? | Compare macro F1 to naive baseline |

**The naive baseline comparison is always worth doing.**

The naive baseline: always predict "Normal." What macro F1 does that achieve?

You might guess 1/3 — Normal F1=1.0, Suspect and Pathological F1=0, average = 0.333. But it is lower than that. When every prediction is "Normal," recall for Normal is 1.0, but precision is only ~0.78 (because Suspect and Pathological samples are also mislabeled as Normal, diluting precision). That gives Normal F1 ≈ 0.876, not 1.0 — so macro F1 ≈ 0.292, not 0.333.

The code cell below confirms this. Your model's macro F1 should be well above ~0.29.

In [ ]:
# Compute naive baseline macro F1
naive_pred = np.zeros(len(y_test), dtype=int)  # always predict "Normal" = class 0
naive_macro = f1_score(y_test, naive_pred, average="macro", zero_division=0)
print(f"Naive baseline macro F1 (always predict 'Normal'): {naive_macro:.3f}")
print(f"Plain XGBoost macro F1                           : {macro_plain:.3f}")
print(f"Balanced XGBoost macro F1                        : {macro_balanced:.3f}")
print()
print(f"Our models are {macro_balanced - naive_macro:.3f} macro F1 points above the naive baseline — clear improvement.")

---

## Putting It All Together

| Concept | What it means |
|---|---|
| Naive baseline macro F1 | Always-Normal model gets macro F1 ≈ 0.29 (not 1/3 — precision < 1.0); real floor to beat |
| Imbalance trap | Simple models default to the majority class, missing rare but important classes |
| Weighted F1 pitfall | Dominated by majority class; hides failures on minority classes |
| Macro F1 | Treats all classes equally — the honest headline metric for imbalanced problems |
| Per-class recall | Direct measure of "is the model catching class C?" |
| `compute_sample_weight("balanced", y)` | Rare classes get high weight so the model has equal incentive to learn each |
| Stratified K-Fold | Preserves class proportions in each fold — essential when minority classes are small |

The story of this unit:

1. Class imbalance distorts model behavior: models default to the majority class when uncertain.
2. Weighted F1 hides this problem. Macro F1 surfaces it. For imbalanced multiclass, macro F1 is your headline metric.
3. Per-class recall for the rare, high-stakes class is your safety check.
4. Sample weights give rare classes equal influence during training — a targeted fix for the imbalance.
5. Nested CV with stratified folds gives an honest performance estimate even when rare classes are very small.

**Full circle:** You now have a complete multiclass toolkit — training, confusion matrix, per-class metrics, averaging strategies, imbalance handling, and honest evaluation. The same patterns that made binary classification tractable extend directly to three or more classes.